# Pothole Risk Model Evaluation

This notebook summarizes the current sample-data evaluation for the Bay Area Pothole AI Tracker. The numbers are useful for portfolio documentation and development checks, but they should not be treated as production validation because the bundled sample dataset is intentionally small.

## Dataset

- Historical labels: `data/sample_bay_area_311_potholes.csv`
- Environmental predictors: `data/sample_bay_area_environmental_features.csv`
- Prediction grid: `predictive-service/data/risk_grid.geojson`

The training script creates positive cells from pothole report locations and nearby negative cells from road/land areas without pothole reports.

In [ ]:
from pathlib import Path
import pandas as pd
metrics_path = Path('model_metrics.csv')
if not metrics_path.exists():
    metrics_path = Path('docs/model_metrics.csv')
metrics = pd.read_csv(metrics_path)
metrics

## Model Comparison

Best sample holdout model by ROC-AUC/F1 ordering: **Random Forest**.

| Model                  |   Accuracy |   Precision |   Recall |    F1 |   ROC-AUC |
|:-----------------------|-----------:|------------:|---------:|------:|----------:|
| Random Forest          |      0.86  |       0.75  |    0.5   | 0.6   |     0.827 |
| Hist Gradient Boosting |      0.86  |       0.8   |    0.444 | 0.571 |     0.743 |
| Decision Tree          |      0.826 |       0.636 |    0.389 | 0.483 |     0.67  |
| Logistic Regression    |      0.558 |       0.25  |    0.556 | 0.345 |     0.641 |
| Weighted Heuristic     |      0.593 |       0.27  |    0.556 | 0.364 |     0.569 |

![Model comparison](figures/model-comparison.svg)

## ROC Curve and Confusion Matrix

The ROC curve shows how well the models separate pothole and non-pothole cells across thresholds. The confusion matrix shows true and false predictions at a 0.5 threshold.

![ROC curve](figures/roc-curve.svg)

![Confusion matrix](figures/confusion-matrix.svg)

## Feature Importance

Feature importance is estimated with permutation importance on the sample holdout set. This helps explain which predictors are most useful for the current model.

![Feature importance](figures/feature-importance.svg)

## Risk Grid Distribution

| risk_band   |   cells |   mean_probability |
|:------------|--------:|-------------------:|
| low         |   17721 |              0.184 |
| medium      |    3159 |              0.514 |
| high        |      66 |              0.759 |

![Risk distribution](figures/risk-distribution.svg)

## Live vs Predicted Comparison

This chart checks where historical pothole reports fall relative to the generated prediction bands. In a production study, the same method should be repeated with future reports that were not used during training.

| risk_band   |   live_reports |   mean_probability |
|:------------|---------------:|-------------------:|
| low         |              5 |              0.31  |
| medium      |             15 |              0.601 |
| high        |              9 |              0.778 |

![Live vs predicted](figures/live-vs-predicted.svg)

## Interpretation

The current results demonstrate that the pipeline can train models, compare them, and generate visual evaluation artifacts. The next academic step is to replace the sample data with larger official historical datasets and rerun this notebook with a time-based validation split.